In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("tech_job_market_2019_2026.csv")
df.head()

In [ ]:
df.shape
df.info()

# conclusions
- posting_date, application_deadline, date_position_filled should be dates but are stored as string
- salary_min_usd and salary_max_usd should be numbers but are stored as string

In [ ]:
df.isnull().sum()

In [ ]:
for col in ['primary_programming_language', 'salary_min_usd', 'salary_max_usd']:
    print(col, ':', df[col].isin(['Unknown', 'TBD', 'Not Disclosed']).sum())

- job_location_city / date_position_filled missing is expected (remote jobs, unfilled jobs)
- salary and language columns have some values written as "Unknown"/"Not Disclosed" instead of blank, these are missing too

In [ ]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate job_id:', df['job_id'].duplicated().sum())
df.drop_duplicates(inplace=True)

- duplicate job_id count is more than duplicate rows, so some job_id are reused for different postings

## fixing categories

In [ ]:
df['sector'].value_counts()

In [ ]:
df['sector'] = df['sector'].str.strip().str.upper()

In [ ]:
df['work_mode'].value_counts()

In [ ]:
df['work_mode'] = df['work_mode'].str.strip().str.lower()
df['work_mode'] = df['work_mode'].str.replace('work from home', 'remote')
df['work_mode'] = df['work_mode'].str.replace('wfh', 'remote')
df['work_mode'] = df['work_mode'].str.replace('on-site', 'onsite')
df['work_mode'] = df['work_mode'].str.replace('in-office', 'onsite')
df['work_mode'].value_counts()

In [ ]:
df['visa_sponsorship_offered'] = df['visa_sponsorship_offered'].replace(
    {"False": 0, "No": 0, "N": 0, "True": 1, "Yes": 1, "Y": 1, "0": 0, "1": 1})
df['layoff_within_12_months_flag'] = df['layoff_within_12_months_flag'].replace(
    {"False": 0, "No": 0, "N": 0, "True": 1, "Yes": 1, "Y": 1, "0": 0, "1": 1})

- binary columns had multiple spellings for true/false, mapped to 0/1

## cleaning salary

In [ ]:
df['salary_min_usd'] = df['salary_min_usd'].astype(str)
df['salary_min_usd'] = df['salary_min_usd'].str.replace('$', '', regex=False)
df['salary_min_usd'] = df['salary_min_usd'].str.replace('USD', '', regex=False)
df['salary_min_usd'] = df['salary_min_usd'].str.replace(',', '', regex=False)
df['salary_min_usd'] = df['salary_min_usd'].replace('Not Disclosed', np.nan)
df['salary_min_usd'] = pd.to_numeric(df['salary_min_usd'], errors='coerce')
df.loc[df['salary_min_usd'] < 0, 'salary_min_usd'] = np.nan

df['salary_max_usd'] = df['salary_max_usd'].astype(str)
df['salary_max_usd'] = df['salary_max_usd'].str.replace('$', '', regex=False)
df['salary_max_usd'] = df['salary_max_usd'].str.replace('USD', '', regex=False)
df['salary_max_usd'] = df['salary_max_usd'].str.replace(',', '', regex=False)
df['salary_max_usd'] = df['salary_max_usd'].replace('Not Disclosed', np.nan)
df['salary_max_usd'] = pd.to_numeric(df['salary_max_usd'], errors='coerce')
df.loc[df['salary_max_usd'] < 0, 'salary_max_usd'] = np.nan

df['salary_min_usd'].describe()

- $ sign, commas, "Not Disclosed" text and -1 placeholder cleaned up, -1 and "Not Disclosed" treated as missing

## distributions

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['salary_min_usd'].dropna(), bins=40)
plt.title('salary min')
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x=df['number_of_applicants'])
plt.title('applicants')
plt.show()

- number_of_applicants has negative values and very high outliers

## bivariate analysis

In [ ]:
plt.figure(figsize=(9,7))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt='.2f')
plt.show()

- offers_extended and offers_accepted highly correlated
- years_experience_min and years_experience_max highly correlated

In [ ]:
plt.figure(figsize=(7,5))
sns.scatterplot(x='offers_extended', y='offers_accepted', data=df, hue='work_mode')
plt.show()

- offers_accepted exceeds offers_extended in some rows, not possible

In [ ]:
(df['offers_accepted'] > df['offers_extended']).sum()

In [ ]:
df.loc[df['offers_extended'] == 0, 'offers_accepted'] = 0
(df['offers_accepted'] > df['offers_extended']).sum()

- fixed the rows where offers_extended was 0, rest left as is since we dont know the true vacancy count

In [ ]:
mask = df['years_experience_min'] > df['years_experience_max']
mask.sum()

In [ ]:
df.loc[mask, ['years_experience_min', 'years_experience_max']] = df.loc[mask, ['years_experience_max', 'years_experience_min']].values

- 315 rows had min > max experience, likely swapped during entry, so swapped them back

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='seniority_level', y='salary_min_usd', data=df,
            order=['Intern', 'Entry', 'Mid', 'Senior', 'Lead', 'Principal'])
plt.title('salary by seniority')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='position_status', y='number_of_interview_rounds', data=df)
plt.title('interview rounds by position status')
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='company_type', y='salary_min_usd', data=df)
plt.title('salary by company type')
plt.show()

## time based analysis

In [ ]:
df['posting_date'] = pd.to_datetime(df['posting_date'], errors='coerce')
df['date_position_filled'] = pd.to_datetime(df['date_position_filled'], errors='coerce')
df['year'] = df['posting_date'].dt.year

In [ ]:
plt.figure(figsize=(9,4))
sns.lineplot(x='year', y='number_of_applicants', hue='company_type', data=df)
plt.show()

## feature engineering

In [ ]:
df['offer_accept_ratio'] = df['offers_accepted'] / df['offers_extended']

In [ ]:
df['time_to_fill_days'] = (df['date_position_filled'] - df['posting_date']).dt.days
df.loc[df['time_to_fill_days'] < 0, 'time_to_fill_days'] = np.nan
df['time_to_fill_days'].describe()

In [ ]:
df['avg_salary_usd'] = (df['salary_min_usd'] + df['salary_max_usd']) / 2

In [ ]:
df['tech_stack_list'] = df['required_tech_stack'].str.split(', ')
df['tech_stack_count'] = df['tech_stack_list'].apply(len)
top_tech = df['tech_stack_list'].explode().value_counts().head(10)

plt.figure(figsize=(8,5))
top_tech.plot(kind='barh')
plt.title('top technologies')
plt.gca().invert_yaxis()
plt.show()

# Conclusions
- salary and date columns needed cleaning before use
- work_mode, sector, visa, layoff columns had inconsistent spellings, fixed
- experience swap and offers_accepted=0 fixed where the reason was clear, other issues (negative applicants, remaining offer mismatch, duplicate job_id) left as is and documented
- salary increases with seniority level
- most extended offers get accepted
- new columns: offer_accept_ratio, time_to_fill_days, avg_salary_usd, tech_stack_count